In [1]:
import datetime as dt
import dask.dataframe as dd
import pandas as pd
from sqlalchemy import select, create_engine
from sqlalchemy.sql.expression import func
from sqlalchemy.sql.expression import literal_column, literal
from sqlalchemy.dialects.postgresql import INTERVAL
from dotenv import load_dotenv
import os
from statsmodels.tsa.stattools import coint
import mc_postgres_db.models as models
from sqlalchemy.orm import Session
from coiled import Cluster
from dask import delayed
from dask.distributed import LocalCluster

load_dotenv()

POSTGRES_URL = os.getenv("POSTGRES_URL")

CLUSTER_TYPE = "local"
N_WORKERS = 10

engine = create_engine(POSTGRES_URL)

In [ ]:
cluster = None
if CLUSTER_TYPE == "local":
    try:
        cluster.close()
    except:
        pass
    cluster = LocalCluster(name="local-cluster", n_workers=N_WORKERS, memory_limit="auto")
elif CLUSTER_TYPE == "coiled":
    cluster = Cluster(
        name="prefect-cluster",
        n_workers=N_WORKERS,
        container="ghcr.io/manning-capital/mc-notebooks:main",
        worker_memory="32GB",
    )

2025-11-02 16:14:58,895 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='127.0.0.1:8787', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/Users/glynfinck/Documents/Repositories/mc-notebooks/.venv/lib/python3.12/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/glynfinck/Documents/Repositories/mc-notebooks/.venv/lib/python3.12/site-packages/tornado/web.py", line 3375, in wrapper
    return method(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/glynfinck/Documents/Repositories/mc-notebooks/.venv/lib/python3.12/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a large

2025-11-02 16:15:06,907 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='127.0.0.1:8787', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/Users/glynfinck/Documents/Repositories/mc-notebooks/.venv/lib/python3.12/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/glynfinck/Documents/Repositories/mc-notebooks/.venv/lib/python3.12/site-packages/tornado/web.py", line 3375, in wrapper
    return method(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/glynfinck/Documents/Repositories/mc-notebooks/.venv/lib/python3.12/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a large

In [3]:
client = cluster.get_client()
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 10
Total threads: 20,Total memory: 40.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:52170,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:52201,Total threads: 2
Dashboard: http://127.0.0.1:52203/status,Memory: 4.00 GiB
Nanny: tcp://127.0.0.1:52173,


In [4]:
# cluster = Cluster(
#     name="prefect-cluster",
#     n_workers=3,
#     container="ghcr.io/manning-capital/mc-notebooks:main",
#     worker_memory="32GB",
# )
# client = cluster.get_client()

In [5]:
date: dt.date = dt.datetime.now(dt.timezone.utc).date() - dt.timedelta(days=1)
end = dt.datetime.combine(date, dt.time.min)
start = end - dt.timedelta(days=30)
start_naive = start.replace(tzinfo=None).replace(second=0, microsecond=0)
end_naive = end.replace(tzinfo=None).replace(second=0, microsecond=0)
print(f"Start: {start}, End: {end}")

Start: 2025-10-02 00:00:00, End: 2025-11-01 00:00:00


In [6]:
max_groups = 100
with Session(engine) as session:
    # Get all provider asset group id(s)
    provider_asset_group_ids = session.scalars(
        select(models.ProviderAssetGroup.id).limit(max_groups)
    ).all()
print(
    f"Provider asset group ids (count: {len(provider_asset_group_ids)}): {provider_asset_group_ids}"
)

Provider asset group ids (count: 100): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100]


In [7]:
# Now select from the subquery
time_frame: dd.DataFrame = dd.read_sql_query(
    select(
        select(
            func.generate_series(
                literal_column(start_naive.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")),
                literal_column(end_naive.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")),
                func.cast(literal("1 minute"), INTERVAL),
            ).label("timestamp")
        ).subquery("time_frame")
    ),
    engine.url.render_as_string(hide_password=False),
    index_col="timestamp",
    bytes_per_chunk="512 MiB",
)
time_frame = time_frame.reset_index()

In [8]:
from dask import delayed
import numpy as np

@delayed
def load_provider_group_members_chunk(group_ids, conn_string):
    """
    Load provider asset group members for a list of group_ids.
    Multiple groups per partition for better control.
    """
    from sqlalchemy import create_engine, select
    import mc_postgres_db.models as models
    import pandas as pd
    
    # Create engine inside function
    engine = create_engine(conn_string)
    
    # Load members for these groups
    df = pd.read_sql(
        select(
            models.ProviderAssetGroupMember.provider_asset_group_id,
            models.ProviderAssetGroupMember.order,
            models.ProviderAssetGroupMember.provider_id,
            models.ProviderAssetGroupMember.from_asset_id,
            models.ProviderAssetGroupMember.to_asset_id,
        ).where(
            models.ProviderAssetGroupMember.provider_asset_group_id.in_(group_ids)
        ),
        engine,
    )
    
    engine.dispose()
    return df


# Split provider_asset_group_ids into chunks
n_partitions = N_WORKERS # Adjust this: 2 partitions means 2-3 groups per partition
group_chunks = np.array_split(provider_asset_group_ids, n_partitions)

# Create delayed tasks - one per chunk of groups
delayed_dfs = []
for group_chunk in group_chunks:
    group_list = group_chunk.tolist()  # Convert to list
    delayed_df = load_provider_group_members_chunk(
        group_list,
        engine.url.render_as_string(hide_password=False)
    )
    delayed_dfs.append(delayed_df)

# Define the schema (meta)
meta = pd.DataFrame({
    'provider_asset_group_id': pd.Series(dtype='int64'),
    'order': pd.Series(dtype='int64'),
    'provider_id': pd.Series(dtype='int64'),
    'from_asset_id': pd.Series(dtype='int64'),
    'to_asset_id': pd.Series(dtype='int64'),
})

# Convert to Dask DataFrame
provider_asset_group_members = dd.from_delayed(delayed_dfs, meta=meta)

In [9]:
time_frame["key"] = 1
provider_asset_group_members["key"] = 1
full_frame = time_frame.merge(
    provider_asset_group_members,
    on="key",
)
full_frame = full_frame.drop(columns=["key"])
full_frame = full_frame.sort_values(by="timestamp")
full_frame = full_frame.set_index("timestamp")

In [10]:
@delayed
def load_market_data_chunk(start_time, end_time, conn_string):
    """
    Load a chunk of market data. This function will be executed by Dask workers.
    """
    from sqlalchemy import create_engine, select
    import mc_postgres_db.models as models
    import pandas as pd

    # Create engine inside function (each worker needs its own)
    engine = create_engine(conn_string)

    # Load the chunk
    df = pd.read_sql(
        select(
            models.ProviderAssetMarket.timestamp,
            models.ProviderAssetMarket.provider_id,
            models.ProviderAssetMarket.from_asset_id,
            models.ProviderAssetMarket.to_asset_id,
            models.ProviderAssetMarket.close,
        )
        .where(models.ProviderAssetMarket.timestamp.between(start_time, end_time))
        .order_by(models.ProviderAssetMarket.timestamp),
        engine,
        index_col="timestamp",
    )

    engine.dispose()
    return df.reset_index()


# Split time range into chunks for parallel loading
time_chunks = pd.date_range(start_naive, end_naive, periods=N_WORKERS + 1)

# Create delayed tasks for each chunk
delayed_dfs = []
for i in range(len(time_chunks) - 1):
    chunk_start = time_chunks[i]
    chunk_end = time_chunks[i + 1]

    delayed_df = load_market_data_chunk(
        chunk_start, chunk_end, engine.url.render_as_string(hide_password=False)
    )
    delayed_dfs.append(delayed_df)

# Define the schema (meta) for the resulting Dask DataFrame
meta = pd.DataFrame(
    {
        "timestamp": pd.Series(dtype="datetime64[ns]"),
        "provider_id": pd.Series(dtype="int64"),
        "from_asset_id": pd.Series(dtype="int64"),
        "to_asset_id": pd.Series(dtype="int64"),
        "close": pd.Series(dtype="float64"),
    }
)

# Convert delayed objects to Dask DataFrame
market_data = dd.from_delayed(delayed_dfs, meta=meta)
market_data = market_data.sort_values(by="timestamp")
market_data = market_data.set_index("timestamp")

In [11]:
full_market_frame = dd.merge_asof(
    full_frame,
    market_data,
    left_index=True,
    right_index=True,
    by=["provider_id", "from_asset_id", "to_asset_id"],
)

In [12]:
# Split out the close for each order and create pairs-trading frame.
close_1 = full_market_frame.loc[
    full_frame["order"] == 1,
    ["provider_asset_group_id", "provider_id", "from_asset_id", "to_asset_id", "close"],
]
close_2 = full_market_frame.loc[
    full_frame["order"] == 2,
    ["provider_asset_group_id", "provider_id", "from_asset_id", "to_asset_id", "close"],
]
pairs_trading_frame: dd.DataFrame = dd.merge(
    close_1,
    close_2,
    on=["timestamp", "provider_asset_group_id"],
    how="inner",
    suffixes=("_1", "_2"),
)

In [13]:
cointegration_p_values = pairs_trading_frame.groupby("provider_asset_group_id")[
    ["close_1", "close_2"]
].apply(
    lambda df: pd.Series(coint(df["close_1"], df["close_2"])[1], index=["p_value"]),
    meta={"p_value": pd.Series([], dtype=float)},
)

In [14]:
cointegration_p_values_computed = cointegration_p_values.compute()
cointegration_p_values_computed

,p_value
provider_asset_group_id,
3,0.038876
20,0.696443
30,0.660388
33,0.116124
54,0.000097
...,...
47,0.706628
63,0.117972
77,0.003242
